In [ ]:
# CELL 0 - Install packages once. (No insightface needed: we convert the ArcFace ONNX file to PyTorch ourselves.)
!pip install -q onnx onnx2torch gdown scikit-learn scikit-image opencv-python-headless tqdm scipy

In [ ]:
# CELL 1 - (same code as Part 1, unchanged) Setup: imports, one Config object holding every setting, and reproducibility. Change numbers here only.
import os, math, time, json
from dataclasses import dataclass
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import deform_conv2d

@dataclass
class Config:
    work_dir: str = "/content/work" if os.path.exists("/content") else "./work"  # data + checkpoints live here
    scale: int = 4              # super-resolution factor (28x28 -> 112x112)
    hr_size: int = 112          # HR crop size = ArcFace input size
    n_val: int = 1000           # fixed validation images (pick the best epoch with these)
    n_test: int = 2000          # fixed test images (used once, at the end)
    feat: int = 48              # feature channels of the network
    blocks: int = 6             # number of residual blocks
    keep_ratio: float = 0.5     # fraction of pixels kept as attention keys (the "sparse" part)
    batch: int = 64
    epochs: int = 30
    lr: float = 2e-4
    num_workers: int = 4        # use 0 on Windows notebooks
    seed: int = 42

cfg = Config()
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)      # reproducible weights and splits
torch.backends.cudnn.benchmark = True                       # faster convolutions for fixed input sizes
print("device:", device, "| work_dir:", cfg.work_dir)

In [ ]:
# CELL 2 - (same code as Part 1, unchanged) Degradation: turns a clean HR face into a CCTV-like LR face (blur -> downsample -> noise -> JPEG).
class Degrader:
    def __init__(self, scale=4, blur_p=0.7, noise=(2, 15), jpeg=(30, 70)):
        self.scale, self.blur_p, self.noise, self.jpeg = scale, blur_p, noise, jpeg
    def __call__(self, hr, rng=None):                                # hr: uint8 HxWx3  ->  lr: uint8 (H/s)x(W/s)x3
        rng = rng or np.random.default_rng()                         # seeded rng => reproducible val/test images
        img = hr
        if rng.random() < self.blur_p:                               # 1) random Gaussian blur (lens / motion blur)
            img = cv2.GaussianBlur(img, (int(rng.choice([3, 5, 7])),) * 2, float(rng.uniform(0.2, 2.0)))
        h, w = hr.shape[:2]
        lr = np.asarray(Image.fromarray(img).resize((w // self.scale, h // self.scale), Image.BICUBIC))  # 2) bicubic downsample (antialiased)
        lr = lr + rng.normal(0, rng.uniform(*self.noise), lr.shape)  # 3) sensor noise
        lr = np.clip(lr, 0, 255).astype(np.uint8)
        ok, buf = cv2.imencode(".jpg", lr, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(*self.jpeg))])    # 4) JPEG compression artifacts
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)

In [ ]:
# CELL 3 - (same code as Part 1, unchanged) Model: sparse sampling conv + adaptive sparse linear attention, stacked in residual blocks. Flags allow ablations.
class SparseSamplingConv(nn.Module):
    """9 learnable sample points per pixel (deformable-style), kept inside a 7x7 window."""
    def __init__(self, c, max_offset=2.0):
        super().__init__()
        self.max_offset = max_offset
        self.offset = nn.Conv2d(c, 18, 3, padding=1)                  # predicts (dy, dx) for each of the 9 points
        nn.init.zeros_(self.offset.weight); nn.init.zeros_(self.offset.bias)   # start exactly as a normal 3x3 conv
        self.weight = nn.Parameter(torch.empty(c, c, 3, 3)); nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(c))
    def forward(self, x):
        off = torch.tanh(self.offset(x)) * self.max_offset            # 1 (grid) + 2 (offset) = 3 px -> 7x7 window
        return deform_conv2d(x, off, self.weight, self.bias, padding=(1, 1))

class AdaptiveSparseLinearAttention(nn.Module):
    """Linear attention O(n) where only the top-k most important pixels act as keys (content-adaptive sparsity)."""
    def __init__(self, c, keep_ratio=0.5):
        super().__init__()
        self.keep_ratio = keep_ratio
        self.qkv = nn.Conv2d(c, 3 * c, 1)
        self.score = nn.Sequential(nn.Conv2d(c, c // 4, 1), nn.GELU(), nn.Conv2d(c // 4, 1, 1))  # per-pixel importance
        self.proj = nn.Conv2d(c, c, 1)
    def key_mask(self, x):
        s = torch.sigmoid(self.score(x)).flatten(1)                   # (N, HW) importance in (0,1)
        k = max(1, int(self.keep_ratio * s.shape[1]))
        hard = torch.zeros_like(s).scatter_(1, s.topk(k, dim=1).indices, 1.0)   # 1 for the top-k pixels, 0 elsewhere
        return hard + s - s.detach()                                  # forward = hard mask, backward = gradient of s (straight-through)
    def forward(self, x):
        n, c, h, w = x.shape
        q, k, v = self.qkv(x).flatten(2).chunk(3, dim=1)              # each (N, C, HW)
        q = F.elu(q) + 1                                              # positive feature map phi(q)
        k = (F.elu(k) + 1) * self.key_mask(x).unsqueeze(1)            # phi(k), zeroed for non-selected pixels
        kv = torch.einsum("ncl,ndl->ncd", k, v)                       # (N,C,C) global summary, computed once -> linear cost
        z = 1.0 / (torch.einsum("ncl,nc->nl", q, k.sum(2)) + 1e-6)    # normalisation per query pixel
        out = torch.einsum("ncd,ncl,nl->ndl", kv, q, z).reshape(n, c, h, w)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, c, use_ssc=True, use_attn=True, keep_ratio=0.5):
        super().__init__()
        self.n1 = nn.GroupNorm(1, c)                                  # LayerNorm-style (no BatchNorm in SR)
        self.spatial = SparseSamplingConv(c) if use_ssc else nn.Conv2d(c, c, 3, padding=1)   # SSC or plain conv (ablation)
        self.attn = AdaptiveSparseLinearAttention(c, keep_ratio) if use_attn else None
        self.n2 = nn.GroupNorm(1, c) if use_attn else None
        self.n3 = nn.GroupNorm(1, c)
        self.ffn = nn.Sequential(nn.Conv2d(c, 2 * c, 1), nn.GELU(), nn.Conv2d(2 * c, c, 1))   # adds non-linearity
    def forward(self, x):
        x = x + self.spatial(self.n1(x))                              # local branch (residual)
        if self.attn is not None: x = x + self.attn(self.n2(x))       # global branch (residual)
        return x + self.ffn(self.n3(x))

class SRNet(nn.Module):
    def __init__(self, cfg, use_ssc=True, use_attn=True):
        super().__init__()
        c, self.scale = cfg.feat, cfg.scale
        self.head = nn.Conv2d(3, c, 3, padding=1)                     # RGB -> features
        self.body = nn.Sequential(*[Block(c, use_ssc, use_attn, cfg.keep_ratio) for _ in range(cfg.blocks)])
        self.tail = nn.Conv2d(c, c, 3, padding=1)
        up = []
        for _ in range(int(math.log2(cfg.scale))): up += [nn.Conv2d(c, 4 * c, 3, padding=1), nn.PixelShuffle(2)]   # 2x, twice
        self.up = nn.Sequential(*up, nn.Conv2d(c, 3, 3, padding=1))
    def forward(self, x):
        f = self.head(x)
        f = self.tail(self.body(f)) + f                               # long skip connection
        return self.up(f) + F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)  # network learns only the residual over bicubic

def n_params(m): return sum(p.numel() for p in m.parameters())

def sanity_check():                                                   # quick test: shapes, gradients and that the sparse mask really is sparse
    m = SRNet(cfg); x = torch.rand(2, 3, 28, 28)
    y = m(x); assert y.shape == (2, 3, 112, 112), y.shape
    y.mean().backward()
    a = m.body[0].attn
    assert a.score[0].weight.grad.abs().sum() > 0, "importance scorer gets no gradient"
    mask = a.key_mask(torch.randn(2, cfg.feat, 28, 28))
    assert abs(mask.round().mean().item() - cfg.keep_ratio) < 0.01, "mask is not top-k sparse"
    print("sanity OK | output", tuple(y.shape), "| params", f"{n_params(m):,}")
sanity_check()

In [ ]:
# CELL 4 - (same code as Part 1, unchanged) Metrics (PSNR/SSIM on the Y channel, as in the papers) and the Trainer class (L1 loss, AdamW, cosine LR, resume).
class Metrics:
    @staticmethod
    def rgb_to_y(x):                                                  # x: BCHW in [0,1] -> luminance Y in [0,1]
        return (16 + 65.481 * x[:, 0] + 128.553 * x[:, 1] + 24.966 * x[:, 2]) / 255.0
    @staticmethod
    def psnr_y(sr, hr, shave):                                        # per-image PSNR, border pixels cropped
        a, b = Metrics.rgb_to_y(sr)[:, shave:-shave, shave:-shave], Metrics.rgb_to_y(hr)[:, shave:-shave, shave:-shave]
        return 10 * torch.log10(1 / ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10))
    @staticmethod
    def ssim_y(sr, hr, shave):
        from skimage.metrics import structural_similarity as ssim
        a, b = Metrics.rgb_to_y(sr)[:, shave:-shave, shave:-shave].numpy(), Metrics.rgb_to_y(hr)[:, shave:-shave, shave:-shave].numpy()
        return torch.tensor([ssim(p, q, data_range=1.0) for p, q in zip(a, b)])

@torch.no_grad()
def evaluate(fn, data, device, shave=4, bs=100, ssim=False):          # fn: any function LR-tensor -> SR-tensor (model or bicubic)
    if isinstance(fn, nn.Module): fn.eval()
    ps, ss = [], []
    for lr, hr in data.batches(bs):
        sr = fn(lr.to(device)).clamp(0, 1).cpu()                      # clamp: valid pixel range
        ps.append(Metrics.psnr_y(sr, hr, shave))
        if ssim: ss.append(Metrics.ssim_y(sr, hr, shave))
    out = {"PSNR_Y": torch.cat(ps).mean().item()}
    if ssim: out["SSIM_Y"] = torch.cat(ss).mean().item()
    return out


In [ ]:
# CELL 5 - QMUL-SurvFace files: download the 3 zips (protocol zip + gallery_p.zip + mated_probe_p.zip) and unzip them.
# If the automatic download fails, download the zips in the browser and put them anywhere inside <work_dir>/qmul_dl, then re-run.
import zipfile
class QMULFetcher:
    FOLDERS = {"main": "13ch6BPaexlKt8gXB_I8aX7p1G3yPm2Bl", "tests": "10eLJIUpMTER_27tEjOvXDo-93FqJAvdB"}   # Google Drive ids
    def __init__(self, cfg):
        self.dl, self.out = Path(cfg.work_dir) / "qmul_dl", Path(cfg.work_dir) / "qmul_data"
        self.dl.mkdir(exist_ok=True); self.out.mkdir(exist_ok=True)
    def download(self):
        import gdown
        for name, gid in self.FOLDERS.items():
            try:
                files = gdown.download_folder(id=gid, output=str(self.dl / name), quiet=False, use_cookies=False)
                if not files: gdown.download(id=gid, output=str(self.dl / f"{name}.zip"), quiet=False)  # main link is a file, not a folder
            except Exception:
                try: gdown.download(id=gid, output=str(self.dl / f"{name}.zip"), quiet=False)                 # ...or a single file
                except Exception: print(f"[{name}] automatic download failed -> download manually into {self.dl}")
    def unzip_all(self):
        for z in sorted(self.dl.rglob("*.zip")):
            marker = self.out / f".done_{z.name}"
            if marker.exists(): continue                                                  # already unzipped
            with zipfile.ZipFile(z) as f:
                f.extractall(self.out, members=[n for n in f.namelist() if "training_set" not in n and "__MACOSX" not in n])  # training images are not needed
            marker.touch(); print("unzipped", z.name)
fetch = QMULFetcher(cfg)
if not any(fetch.dl.rglob("*.zip")): fetch.download()
fetch.unzip_all()

In [ ]:
# CELL 6 - ArcFace face recogniser: PRETRAINED and FROZEN (it only measures our result, it is never trained). Converted to PyTorch so it runs batched on the GPU.
import urllib.request, zipfile
from onnx2torch import convert
import onnx
def to_tensor(a):                                                     # HWC uint8 -> CHW float in [0,1]
    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div(255)

class ArcFace(nn.Module):
    URL = "https://github.com/deepinsight/insightface/releases/download/model-zoo/buffalo_l.zip"   # pack that contains w600k_r50.onnx
    def __init__(self, work_dir):
        super().__init__()
        onnx_path = Path(work_dir) / "w600k_r50.onnx"
        if not onnx_path.exists():                                    # download once, keep only the recognition model
            zp = Path(work_dir) / "buffalo_l.zip"; urllib.request.urlretrieve(self.URL, zp)
            with zipfile.ZipFile(zp) as z: onnx_path.write_bytes(z.read([n for n in z.namelist() if n.endswith("w600k_r50.onnx")][0]))
        self.net = convert(onnx.load(str(onnx_path))).eval()         # in-memory conversion avoids Windows temporary-file locking
        for p in self.net.parameters(): p.requires_grad_(False)       # frozen
    def forward(self, x):                                             # x: RGB in [0,1], shape B x 3 x 112 x 112
        return F.normalize(self.net((x - 0.5) / 0.5), dim=1)          # insightface normalisation -> unit-length 512-d identity vector
arcface = ArcFace(cfg.work_dir).to(device).eval()
print("ArcFace ready on", device)

In [ ]:
# CELL 7 - Read the QMUL protocol: gallery/probe images + identities, and the verification pairs (same person / different person).
import scipy.io as sio
def unwrap(c):                                                        # MATLAB cell array -> plain python string
    while isinstance(c, np.ndarray): c = c[0]
    return str(c)

class QMULData:
    def __init__(self, work_dir):
        base = Path(work_dir) / "qmul_data"
        hit = next((p.parent for p in base.rglob("Face_Verification_Test_Set") if "__MACOSX" not in str(p)), None)
        if hit is None: raise FileNotFoundError(f"protocol folder not found under {base} - run CELL 5 / check the zips")
        self.root, self.paths = hit, {}
        for pool in ["gallery", "mated_probe", "unmated_probe", "verification_images"]:  # include verification-specific crops
            for d in base.rglob(pool):
                if d.is_dir() and "__MACOSX" not in str(d):
                    for f in os.listdir(d): self.paths.setdefault(f, d / f)   # filename -> full path
        ident, verif = hit / "Face_Identification_Test_Set", hit / "Face_Verification_Test_Set"
        g, p = sio.loadmat(ident / "gallery_img_ID_pairs.mat"), sio.loadmat(ident / "mated_probe_img_ID_pairs.mat")
        self.gal_files, self.gal_ids = [unwrap(x) for x in g["gallery_set"].flatten()], g["gallery_ids"].flatten().astype(int)
        self.pro_files, self.pro_ids = [unwrap(x) for x in p["mated_probe_set"].flatten()], p["mated_probe_ids"].flatten().astype(int)
        pos = sio.loadmat(verif / "positive_pairs_names.mat")["positive_pairs_names"]
        neg = sio.loadmat(verif / "negative_pairs_names.mat")["negative_pairs_names"]
        self.pairs = [(unwrap(a), unwrap(b)) for a, b in pos] + [(unwrap(a), unwrap(b)) for a, b in neg]
        self.pair_labels = np.array([1] * len(pos) + [0] * len(neg))  # 1 = same person, 0 = different
        need = set(self.gal_files) | set(self.pro_files) | {f for ab in self.pairs for f in ab}
        self.missing = sorted(need - set(self.paths))
        self.files = sorted(need - set(self.missing))                 # every image we embed (each only once)
        print(f"gallery {len(self.gal_files)} | probes {len(self.pro_files)} | pairs {len(self.pairs)} | images {len(self.files)} | MISSING {len(self.missing)}")
    def size_stats(self, n=2000):                                     # tells us if 28x28 is a sensible LR size for these faces
        s = np.array([Image.open(self.paths[f]).size for f in self.files[:n]])
        print("native size (w,h): median", np.median(s, 0), "| min", s.min(0), "| max", s.max(0))
qmul = QMULData(cfg.work_dir); qmul.size_stats()

In [ ]:
# CELL 8 - Embeddings: each image is read ONCE, turned into 4 inputs (the "conditions"), passed through ArcFace, and cached on disk.
from tqdm.auto import tqdm
bicubic = lambda x: F.interpolate(x, scale_factor=cfg.scale, mode="bicubic", align_corners=False)

def load_sr(tag, **kw):                                               # load a Part-1 checkpoint (best epoch chosen on validation)
    m = SRNet(cfg, **kw).to(device)
    m.load_state_dict(torch.load(Path(cfg.work_dir) / "ckpt" / f"{tag}_best.pt", map_location=device)["model"])
    return m.eval()
sr_models = {"baseline_conv": load_sr("baseline_conv", use_ssc=False, use_attn=False), "ours_ssc_attn": load_sr("ours_ssc_attn")}

class ImageSet(Dataset):
    def __init__(self, paths, lr_size): self.paths, self.lr = paths, lr_size
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return to_tensor(np.asarray(img.resize((112, 112), Image.BICUBIC))), to_tensor(np.asarray(img.resize((self.lr, self.lr), Image.BICUBIC)))  # (native->112, tiny LR input)

class EmbeddingBank:
    CONDS = ["native_bicubic", "lr_bicubic", "baseline_conv", "ours_ssc_attn"]
    def __init__(self, models, arc, cfg): self.models, self.arc, self.cfg = models, arc, cfg
    @staticmethod
    def fit112(v): return v if v.shape[-1] == 112 else F.interpolate(v, size=(112, 112), mode="bicubic", align_corners=False)
    @torch.no_grad()
    def compute(self, files, paths, tag, lr_size=28, bs=128):
        cdir = Path(self.cfg.work_dir) / "emb"; cdir.mkdir(exist_ok=True); f = cdir / f"{tag}.npz"
        if f.exists():
            z = np.load(f)
            if len(z["files"]) == len(files) and z["files"][0] == files[0] and z["files"][-1] == files[-1]:
                print("loaded cached embeddings:", tag); return {c: z[c] for c in self.CONDS}
        loader = DataLoader(ImageSet([paths[x] for x in files], lr_size), batch_size=bs, num_workers=self.cfg.num_workers)
        out = {c: [] for c in self.CONDS}
        for native, lr in tqdm(loader):
            native, lr = native.to(device), lr.to(device)
            views = {"native_bicubic": native,                        # naive pipeline: resize the original straight to 112
                     "lr_bicubic": self.fit112(bicubic(lr)),          # same tiny LR input, plain bicubic enlargement
                     **{k: self.fit112(m(lr)) for k, m in self.models.items()}}   # same tiny LR input, super-resolved
            for c, v in views.items(): out[c].append(self.arc(v.clamp(0, 1)).cpu())
        emb = {c: torch.cat(v).numpy() for c, v in out.items()}
        np.savez(f, files=np.array(files), **emb); return emb

In [ ]:
# CELL 9 - Recognition metrics: verification (10-fold accuracy, AUC, TAR@FAR) and closed-set identification (Rank-1/5), with 95% bootstrap CIs.
from sklearn.metrics import roc_auc_score, roc_curve

class VerificationEval:
    def __init__(self, n_folds=10, boots=200, seed=0): self.k, self.B, self.seed = n_folds, boots, seed
    @staticmethod
    def best_threshold(s, y):                                         # threshold with the highest accuracy on TRAIN folds only
        o = np.argsort(s); s, y = s[o], y[o]
        neg_below = np.concatenate([[0], np.cumsum(y == 0)]); pos_from = (y == 1).sum() - np.concatenate([[0], np.cumsum(y == 1)])
        i = int(np.argmax(neg_below + pos_from))                      # accept pairs with sorted index >= i
        return (s[i - 1] + s[i]) / 2 if 0 < i < len(s) else (s[0] - 1e-6 if i == 0 else s[-1] + 1e-6)
    def __call__(self, ea, eb, y, folds=None):
        s, rng = (ea * eb).sum(1), np.random.default_rng(self.seed)   # cosine similarity of unit vectors
        if folds is None:                                             # stratified random folds
            folds = np.zeros(len(y), int)
            for c in (0, 1): idx = np.where(y == c)[0]; folds[rng.permutation(idx)] = np.arange(len(idx)) % self.k
        accs = [((s[folds == f] >= self.best_threshold(s[folds != f], y[folds != f])) == y[folds == f]).mean() for f in range(self.k)]
        fpr, tpr, _ = roc_curve(y, s)
        aucs = [roc_auc_score(y[b], s[b]) for b in (rng.integers(0, len(y), len(y)) for _ in range(self.B)) if 0 < y[b].sum() < len(b)]
        return {"acc": float(np.mean(accs)), "acc_std": float(np.std(accs)), "auc": float(roc_auc_score(y, s)),
                "auc_lo": float(np.percentile(aucs, 2.5)), "auc_hi": float(np.percentile(aucs, 97.5)),
                "tar_far1": float(np.interp(0.01, fpr, tpr)), "tar_far10": float(np.interp(0.1, fpr, tpr))}   # TAR at FAR = 1% and 10%

class IdentificationEval:
    def __init__(self, device, chunk=4096, boots=200, seed=0): self.dev, self.chunk, self.B, self.seed = device, chunk, boots, seed
    @torch.no_grad()
    def __call__(self, eg, ig, ep, ip):                               # gallery emb/ids, probe emb/ids
        classes = np.unique(ig); gi = torch.from_numpy(np.searchsorted(classes, ig))
        T = F.normalize(torch.zeros(len(classes), eg.shape[1]).index_add_(0, gi, torch.from_numpy(eg)), dim=1).to(self.dev)   # one mean template per identity
        keep = np.isin(ip, classes); ep, pi = ep[keep], torch.from_numpy(np.searchsorted(classes, ip[keep]))
        h1, h5 = [], []
        for i in range(0, len(ep), self.chunk):
            top = (torch.from_numpy(ep[i:i + self.chunk]).to(self.dev) @ T.T).topk(min(5, len(classes)), dim=1).indices.cpu()   # best 5 identities per probe
            t = pi[i:i + self.chunk]; h1.append(top[:, 0] == t); h5.append((top == t[:, None]).any(1))
        h1, h5, rng = torch.cat(h1).numpy(), torch.cat(h5).numpy(), np.random.default_rng(self.seed)
        b1 = [h1[rng.integers(0, len(h1), len(h1))].mean() for _ in range(self.B)]
        return {"rank1": float(h1.mean()), "rank1_lo": float(np.percentile(b1, 2.5)), "rank1_hi": float(np.percentile(b1, 97.5)),
                "rank5": float(h5.mean()), "n_probes": int(len(h1)), "n_ids": int(len(classes))}

def show(results, title):                                             # pretty table
    print(f"\n{title}\n{'condition':16s}{'ver acc %':>13s}{'AUC':>8s}{'TAR@1%':>8s}{'TAR@10%':>9s}{'Rank-1 %':>10s}{'Rank-5 %':>10s}")
    for c, r in results.items():
        v, i = r["ver"], r.get("ident")
        print(f"{c:16s}{100 * v['acc']:>8.2f}+-{100 * v['acc_std']:.1f}{v['auc']:>8.4f}{v['tar_far1']:>8.3f}{v['tar_far10']:>9.3f}" +
              (f"{100 * i['rank1']:>10.2f}{100 * i['rank5']:>10.2f}" if i else ""))

In [ ]:
# CELL 10 - REAL-WORLD TEST (no clean HR exists): QMUL-SurvFace. Same images, 4 conditions -> verification + identification.
emb = EmbeddingBank(sr_models, arcface, cfg).compute(qmul.files, qmul.paths, tag="qmul")     # takes a few minutes on a GPU
ix = {f: i for i, f in enumerate(qmul.files)}
look = lambda fs: np.array([ix.get(f, -1) for f in fs])               # filename -> embedding row (-1 = image missing)
ia, ib = look([a for a, b in qmul.pairs]), look([b for a, b in qmul.pairs]); ok = (ia >= 0) & (ib >= 0)
gi, pi_ = look(qmul.gal_files), look(qmul.pro_files); gk, pk = gi >= 0, pi_ >= 0
print(f"pairs used {ok.sum()}/{len(ok)} | gallery used {gk.sum()}/{len(gk)} | probes used {pk.sum()}/{len(pk)}")
qres = {}
for c in EmbeddingBank.CONDS:
    E = emb[c]
    qres[c] = {"ver": VerificationEval()(E[ia[ok]], E[ib[ok]], qmul.pair_labels[ok]),
               "ident": IdentificationEval(device)(E[gi[gk]], qmul.gal_ids[gk], E[pi_[pk]], qmul.pro_ids[pk])}
show(qres, "QMUL-SurvFace (real surveillance faces)")
print("95% CI  AUC / Rank-1:", {c: (round(r["ver"]["auc_lo"], 3), round(r["ver"]["auc_hi"], 3), round(r["ident"]["rank1_lo"], 3), round(r["ident"]["rank1_hi"], 3)) for c, r in qres.items()})
json.dump(qres, open(Path(cfg.work_dir) / "part2a_qmul.json", "w"), indent=2)

In [ ]:
# CELL 11 - CONTROLLED TEST on LFW: clean HR faces exist, so we can compare HR (ceiling) > SR > plain bicubic (floor) on faces never seen in training.
class LFWDegradedExperiment:
    def __init__(self, models, arc, degrader): self.models, self.arc, self.deg = models, arc, degrader
    def load(self, data_home):                                        # 6000 standard pairs, centre 112x112 crop of the funneled images
        from sklearn.datasets import fetch_lfw_pairs
        d = fetch_lfw_pairs(subset="10_folds", funneled=True, color=True, resize=1.0, slice_=(slice(69, 181), slice(69, 181)), data_home=str(data_home))
        x = d.pairs * 255 if d.pairs.max() <= 1.5 else d.pairs        # sklearn scales pixels to [0,1]
        return np.clip(np.rint(x), 0, 255).astype(np.uint8), d.target
    @torch.no_grad()
    def run(self, pairs, target, bs=200):
        n, hr = len(pairs), pairs.reshape(-1, 112, 112, 3)            # image order: pair0-a, pair0-b, pair1-a, ...
        lr = np.stack([self.deg(h, np.random.default_rng(5000 + i)) for i, h in enumerate(hr)])   # training degradation, fixed seeds
        tt = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255)
        conds = ["hr_ceiling", "lr_bicubic", "baseline_conv", "ours_ssc_attn"]
        embs, psnr = {c: [] for c in conds}, {c: [] for c in conds[1:]}
        for i in range(0, len(hr), bs):
            h, l = tt(hr[i:i + bs]).to(device), tt(lr[i:i + bs]).to(device)
            outs = {"hr_ceiling": h, "lr_bicubic": bicubic(l), **{k: m(l) for k, m in self.models.items()}}
            for c, o in outs.items():
                o = o.clamp(0, 1); embs[c].append(self.arc(o).cpu())
                if c != "hr_ceiling": psnr[c].append(Metrics.psnr_y(o.cpu(), h.cpu(), 4))     # pixel quality vs the true HR face
        folds, res = np.arange(n) * 10 // n, {}                      # the official 10 LFW folds
        for c in conds:
            E = torch.cat(embs[c]).numpy()
            res[c] = {"ver": VerificationEval()(E[0::2], E[1::2], np.asarray(target), folds), "psnr_y": float(torch.cat(psnr[c]).mean()) if c in psnr else None}
        return res
lfw = LFWDegradedExperiment(sr_models, arcface, Degrader(cfg.scale))
pairs, target = lfw.load(Path(cfg.work_dir) / "lfw")                  # downloads ~230 MB the first time
lres = lfw.run(pairs, target); show(lres, "LFW pairs, degraded to 28x28 (HR = upper bound)")
print("PSNR-Y vs true HR:", {c: round(r["psnr_y"], 2) for c, r in lres.items() if r["psnr_y"]})
json.dump(lres, open(Path(cfg.work_dir) / "part2a_lfw.json", "w"), indent=2)